<p style="text-align:center">
    <img src="https://media.licdn.com/dms/image/v2/D560BAQGRKQWaOldXvg/company-logo_200_200/company-logo_200_200/0/1723551442229/cambo_ai_logo?e=2147483647&v=beta&t=FrK5k676UJMjoOULl4R3euChg1VvClew28fxNscSd9E" width="80" alt="CamboAI Logo">
</p>

# **I. Objectives**

**Estimated time needed: 30 minutes**

**After completing this lab you will be able to:**
- Know how to use sql in MySQL & Postgresql Databases.
- Perform **CRUD** Operation in SQL.
- Perform ETL From **Source** (MySQL) to **Destination** (PostgreSQL) Database.

# **II. Import Libraries**

In [2]:
# install psycopg2, pymysql libraries
%pip install psycopg2 pymysql mysql-connector-python --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
# import libraries
import pymysql
import psycopg2
import mysql.connector
import pandas as pd

# filter warning
import warnings
warnings.filterwarnings('ignore')

# **III. Extract the Data from Source**

In [4]:
# step1 : function to extract data from MySQL
def extract_from_mysql(mysql_config, query):
    connection = mysql.connector.connect(**mysql_config)
    df = pd.read_sql(query, connection)
    connection.close()
    return df

# **IV. Transform the Data**

In [5]:
# step 2: transform the data
def transform_data(df):
    # rename columns to lowercase
    df.columns = [col.lower() for col in df.columns]

    # other transformations can be done here
    return df

# **V. Load the Data Into Target**

In [6]:
# step 3: function to load data into PostgreSQL with truncate
def load_to_postgres(postgres_config, df, table_name):
    # connect to PostgreSQL database
    connection = psycopg2.connect(
        host=postgres_config['host'],
        user=postgres_config['user'],
        password=postgres_config['password'],
        database=postgres_config['database']
    )

    cursor = connection.cursor()

    # step 1: truncate the destination table
    truncate_query = f'TRUNCATE TABLE {table_name};'
    cursor.execute(truncate_query)

    # step 2: load data into PostgreSQL
    for i, row in df.iterrows():
        insert_query = f'''
        INSERT INTO {table_name} ({", ".join(df.columns)})
        VALUES ({", ".join(["%s"] * len(df.columns))})
        '''
        cursor.execute(insert_query, tuple(row))

    connection.commit()

    # close PostgreSQL connection
    cursor.close()
    connection.close()

# **VI. ETL Operations**

In [8]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

# main ELT function
def run_elt():
    mysql_config = {
    'host': os.getenv('MYSQL_HOST', 'localhost'),
    'user': os.getenv('MYSQL_USER'),
    'password': os.getenv('MYSQL_PASSWORD', ''),
    'database': os.getenv('MYSQL_DATABASE')
    }

    # New dynamic PostgreSQL configuration
    postgres_config = {
        'host': os.getenv('POSTGRES_HOST', 'localhost'),
        'user': os.getenv('POSTGRES_USER'),
        'password': os.getenv('POSTGRES_PASSWORD', ''),
        'database': os.getenv('POSTGRES_DATABASE')
    }

    # SQL query to extract data from MySQL
    query = 'SELECT * FROM tbl_students;'

    # step 1: extract data from MySQL
    df = extract_from_mysql(mysql_config, query)

    # step 2: load the extracted data into PostgreSQL
    load_to_postgres(postgres_config, df, 'public.tbl_students')

    print("ELT process completed successfully.")

# run the ELT process
if __name__ == '__main__':
    run_elt()

ELT process completed successfully.


# Excellent! You have just completed this Notebook.

This notebook was created by [Chanthan DINA](https://www.linkedin.com/in/chanthandina/). I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!

## <h3 align="center"> &#169; CamboAI. All rights reserved. <h3/>